# Capítulo 1 – Análisis Exploratorio de Datos (EDA)
## Sistema de Bicicletas Compartidas – Dataset `hour.csv`

En este capítulo realizamos un análisis exploratorio inicial del conjunto de datos
horario del sistema de bicicletas compartidas. El objetivo es entender la estructura
de los datos, las variables disponibles y sus relaciones básicas con la demanda
de bicicletas (`cnt`).

Este notebook está diseñado para formar parte del libro Jupyter-Book del proyecto
de regresión lineal múltiple avanzada.


## 1. Carga de librerías y datos

Primero importamos las librerías necesarias y cargamos el archivo `hour.csv` 
desde la carpeta `data/` ubicada en el directorio raíz del proyecto.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración general de gráficos
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11


In [ ]:
# Ruta relativa al archivo de datos desde la carpeta 'book'
data_path = "../data/hour.csv"

df = pd.read_csv(data_path)

df.head()

### 1.1. Estructura general del dataset

Revisamos el número de observaciones, variables y tipos de datos para tener una
primera idea de la información disponible.


In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

### 1.2. Conversión de variables categóricas

Algunas variables son códigos enteros que representan categorías (por ejemplo,
estación, mes, hora, tipo de día, condiciones climáticas, etc.). Las convertimos
al tipo `category` para facilitar su manejo en análisis posteriores.


In [ ]:
categorical_cols = [
    "season", "yr", "mnth", "hr", "holiday",
    "weekday", "workingday", "weathersit"
]

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")

df[categorical_cols].head()

## 2. Variable objetivo: demanda de bicicletas (`cnt`)

La variable de interés principal es `cnt`, que representa el **número total de
alquileres de bicicletas** (usuarios casuales + registrados) en cada hora.

A continuación analizamos su distribución.


In [ ]:
df["cnt"].describe()

In [ ]:
fig, ax = plt.subplots()
ax.hist(df["cnt"], bins=30, edgecolor="black")
ax.set_title("Distribución de la demanda horaria (cnt)")
ax.set_xlabel("cnt")
ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()

## 3. Variables de uso relacionadas: `casual` y `registered`

Además de `cnt`, el dataset incluye:

- `casual`: usuarios que alquilan bicicletas de manera ocasional.
- `registered`: usuarios registrados en el sistema.

Analizamos su distribución y relación con `cnt`.


In [ ]:
df[["casual", "registered"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(df["casual"], bins=30, edgecolor="black")
axes[0].set_title("Distribución de usuarios casuales")
axes[0].set_xlabel("casual")
axes[0].set_ylabel("Frecuencia")

axes[1].hist(df["registered"], bins=30, edgecolor="black")
axes[1].set_title("Distribución de usuarios registrados")
axes[1].set_xlabel("registered")
axes[1].set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()

In [ ]:
df[["cnt", "casual", "registered"]].corr()

## 4. Variables ambientales

El dataset incluye variables climáticas normalizadas:

- `temp`: temperatura normalizada.
- `atemp`: sensación térmica normalizada.
- `hum`: humedad relativa.
- `windspeed`: velocidad del viento normalizada.

Analizamos su distribución y posibles relaciones con la demanda `cnt`.


In [ ]:
env_vars = ["temp", "atemp", "hum", "windspeed"]
df[env_vars].describe()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
axes = axes.ravel()
for i, col in enumerate(env_vars):
    axes[i].hist(df[col], bins=30, edgecolor="black")
    axes[i].set_title(f"Distribución de {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
axes = axes.ravel()
for i, col in enumerate(env_vars):
    axes[i].scatter(df[col], df["cnt"], alpha=0.3)
    axes[i].set_title(f"cnt vs {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("cnt")
plt.tight_layout()
plt.show()

## 5. Variables temporales

Las variables temporales permiten estudiar patrones por hora, día de la semana,
estación y año:

- `hr`: hora del día (0–23).
- `weekday`: día de la semana.
- `workingday`: indicador de día laboral.
- `season`: estación del año.
- `yr`: año (por ejemplo, 0 = 2011, 1 = 2012).

A continuación, exploramos cómo varía `cnt` a lo largo de estas dimensiones.


In [ ]:
# Promedio de 'cnt' por hora del día
cnt_by_hr = df.groupby("hr")["cnt"].mean()
fig, ax = plt.subplots()
cnt_by_hr.plot(kind="bar", ax=ax)
ax.set_title("Promedio de cnt por hora del día")
ax.set_xlabel("hr")
ax.set_ylabel("cnt promedio")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Promedio de 'cnt' por día laboral vs no laboral
if "workingday" in df.columns:
    fig, ax = plt.subplots()
    df.groupby("workingday")["cnt"].mean().plot(kind="bar", ax=ax)
    ax.set_title("Promedio de cnt: workingday vs no workingday")
    ax.set_xlabel("workingday")
    ax.set_ylabel("cnt promedio")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# Promedio de 'cnt' por estación
if "season" in df.columns:
    fig, ax = plt.subplots()
    df.groupby("season")["cnt"].mean().plot(kind="bar", ax=ax)
    ax.set_title("Promedio de cnt por estación")
    ax.set_xlabel("season")
    ax.set_ylabel("cnt promedio")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 6. Matriz de correlación

Calculamos la matriz de correlación para un subconjunto de variables numéricas
de interés, incluyendo la demanda de bicicletas (`cnt`). Esto nos permitirá
identificar relaciones lineales potencialmente útiles para el modelo.


In [ ]:
numeric_cols = [
    col for col in df.columns
    if df[col].dtype != "category" and col not in ["instant"]
]
corr = df[numeric_cols].corr()
corr["cnt"].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Matriz de correlación (variables numéricas)")
plt.tight_layout()
plt.show()

## 7. Variables candidatas para el modelo de regresión

A partir del análisis exploratorio, de la interpretación de las variables y de
las correlaciones con `cnt`, una primera selección de **variables candidatas**
para el modelo de regresión lineal múltiple podría incluir:

- Variables de uso:
  - `registered`, `casual` (según el objetivo, podrían ser explicativas o parte
    de otros modelos intermedios).

- Variables ambientales:
  - `temp`, `atemp`, `hum`, `windspeed`.

- Variables temporales y de calendario:
  - `hr`, `weekday`, `workingday`, `season`, `yr`, `weathersit`.

En los siguientes capítulos refinaremos esta selección mediante criterios
estadísticos (correlaciones, multicolinealidad, significancia, validación
cruzada, etc.) y consideraciones de interpretabilidad.


## 8. Conclusión del EDA inicial

En este capítulo:

- Exploramos la estructura del dataset `hour.csv`.
- Analizamos la distribución de la demanda `cnt` y las variables de uso.
- Estudiamos la relación de `cnt` con variables ambientales y temporales.
- Identificamos un conjunto inicial de variables candidatas para el modelo.

En el próximo capítulo abordaremos la **preparación y transformación de datos**,
incluyendo codificación de variables categóricas, posibles transformaciones y
la construcción de matrices de diseño para los modelos de regresión.
